<a href="https://colab.research.google.com/github/Ayseatci/DI725_Project/blob/dev/notebooks/03_phase2/SWIN_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Image Model: SWIN V2

In [2]:
!pip install -q timm transformers accelerate

In [3]:
!pip install -q timm transformers accelerate wandb open_clip_torch huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.6 MB/s eta 0:00:00


In [4]:
import os
import re
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import timm
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

from torchvision import transforms
import matplotlib.pyplot as plt

import wandb
import open_clip
from huggingface_hub import hf_hub_download

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!unzip -q /content/drive/MyDrive/DI725_project_dataset_nomask.zip

In [6]:
LOCAL_DATA_ROOT = "/content/DI725_project_dataset_nomask"
LOCAL_CSV_PATH = os.path.join(LOCAL_DATA_ROOT, "captions.csv")
LOCAL_IMAGE_DIR = os.path.join(LOCAL_DATA_ROOT, "images")

os.makedirs(LOCAL_DATA_ROOT, exist_ok=True)

CSV_PATH = LOCAL_CSV_PATH
IMAGE_DIR = LOCAL_IMAGE_DIR

print("CSV:", CSV_PATH)
print("Images:", IMAGE_DIR)
print("Number of images:", len(os.listdir(IMAGE_DIR)))

CSV: /content/DI725_project_dataset_nomask/captions.csv
Images: /content/DI725_project_dataset_nomask/images
Number of images: 10000


In [7]:
CONFIG = {
    "sample_size": 10000,
    "img_size": 256,

    "batch_size": 32,
    "epochs": 15,
    "lr": 1e-4,
    "weight_decay": 1e-4,

    "model_name": "swinv2_tiny_window8_256",

    "early_stopping_patience": 3,
    "grad_clip": 1.0,

    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed": 42,

    "text_scale": 0.7
}

TARGET_COLS = ["Tree", "Shrub", "Grass", "Crop", "Built-up", "Barren", "Water"]

In [8]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(CONFIG["seed"])

Data Loading and Text Cleaning

In [9]:
df = pd.read_csv(CSV_PATH)

TEXT_COLUMNS_RAW = [
    "hybrid_gemma3-4b",
    "text_qwen3-4b",
    "vision_gemma3-4b"
]

def clean_caption_no_numbers(text):
    text = str(text).lower()

    # remove percentages: 53%, 53 percent, 53 percentage
    text = re.sub(r"\b\d+(\.\d+)?\s*%|\b\d+(\.\d+)?\s*percent(age)?", "", text)

    # remove standalone numbers
    text = re.sub(r"\b\d+(\.\d+)?\b", "", text)

    leakage_words = [
        "covering", "coverage", "accounts for", "occupies",
        "making up", "comprises", "constitutes", "represents",
        "estimated", "approximately", "around", "about"
    ]

    for word in leakage_words:
        text = text.replace(word, "")

    text = re.sub(r"[%(),:;]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


for col in TEXT_COLUMNS_RAW:
    clean_col = col.replace("-", "_") + "_clean"
    df[clean_col] = df[col].apply(clean_caption_no_numbers)

    print(clean_col)
    print(
        "Remaining numeric leakage:",
        df[clean_col].str.contains(
            r"\d+\s*%|\d+\s*percent|\b\d+(\.\d+)?\b",
            regex=True,
            case=False
        ).sum()
    )

hybrid_gemma3_4b_clean
Remaining numeric leakage: 0


/tmp/ipykernel_2252/2462767828.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[clean_col].str.contains(


text_qwen3_4b_clean
Remaining numeric leakage: 0


/tmp/ipykernel_2252/2462767828.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[clean_col].str.contains(


vision_gemma3_4b_clean
Remaining numeric leakage: 0


/tmp/ipykernel_2252/2462767828.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[clean_col].str.contains(


In [10]:
df["hybrid_gemma3_4b_clean"].head(10)

,hybrid_gemma3_4b_clean
0,the image depicts a landscape dominated by ext...
1,the image depicts a largely arid landscape dom...
2,the image depicts a landscape dominated by ext...
3,the image depicts a valley dominated by dense ...
4,the image depicts a coastal area dominated by ...
5,the image depicts a landscape dominated by cul...
6,the image depicts a landscape dominated by a r...
7,the image depicts a hilly landscape dominated ...
8,the image depicts a landscape dominated by ext...
9,the image depicts a landscape dominated by ext...


In [11]:
df["hybrid_gemma3_4b_clean"].str.contains(
    r"\d+\s*%|\d+\s*percent|\b\d+(\.\d+)?\b",
    regex=True,
    case=False
).sum()

/tmp/ipykernel_2252/1445695331.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["hybrid_gemma3_4b_clean"].str.contains(


np.int64(0)

In [12]:
df["text_qwen3_4b_clean"].str.contains(
    r"\d+\s*%|\d+\s*percent|\b\d+(\.\d+)?\b",
    regex=True,
    case=False
).sum()

/tmp/ipykernel_2252/895902091.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["text_qwen3_4b_clean"].str.contains(


np.int64(0)

Stratified Sampling and Split

In [13]:
needed_cols = (
    ["filename"]
    + TARGET_COLS
    + ["hybrid_gemma3_4b_clean", "text_qwen3_4b_clean", "vision_gemma3_4b_clean"]
)

df = df.dropna(subset=needed_cols).copy()

df["dominant_class"] = df[TARGET_COLS].idxmax(axis=1)

# If available rows <= sample_size, use all rows
if len(df) <= CONFIG["sample_size"]:
    sample_df = df.sample(frac=1.0, random_state=CONFIG["seed"]).reset_index(drop=True)
else:
    sample_df, _ = train_test_split(
        df,
        train_size=CONFIG["sample_size"],
        stratify=df["dominant_class"],
        random_state=CONFIG["seed"]
    )
    sample_df = sample_df.reset_index(drop=True)

print("Sample size:", len(sample_df))

sample_df = sample_df.reset_index(drop=True)

train_df, temp_df = train_test_split(
    sample_df,
    test_size=0.20,
    stratify=sample_df["dominant_class"],
    random_state=CONFIG["seed"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["dominant_class"],
    random_state=CONFIG["seed"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Sample size: 10000
Train: 8000
Val: 1000
Test: 1000


In [14]:
def check_target_distribution(train_df, val_df, test_df, target_cols):
    summary = []

    for name, data in [
        ("train", train_df),
        ("val", val_df),
        ("test", test_df)
    ]:
        row = {"split": name, "n": len(data)}

        for col in target_cols:
            row[f"{col}_mean"] = data[col].mean()
            row[f"{col}_std"] = data[col].std()

        summary.append(row)

    return pd.DataFrame(summary)

dist_summary = check_target_distribution(train_df, val_df, test_df, TARGET_COLS)
dist_summary

,split,n,Tree_mean,Tree_std,Shrub_mean,Shrub_std,Grass_mean,Grass_std,Crop_mean,Crop_std,Built-up_mean,Built-up_std,Barren_mean,Barren_std,Water_mean,Water_std
0,train,8000,28.8445,35.170320,0.828875,4.064469,45.367375,33.938682,17.95475,30.544088,1.139625,5.456130,4.192875,11.363640,1.672,9.244490
1,val,1000,28.8610,35.001154,0.818000,3.981297,45.064000,34.437681,18.15700,30.993789,1.190000,4.985053,4.181000,11.631231,1.729,9.926837
2,test,1000,28.1980,34.791741,0.941000,4.581103,45.318000,34.315075,18.14000,30.799334,1.095000,4.951615,4.508000,12.302898,1.800,9.461793


In [15]:
dominant_dist = pd.DataFrame({
    "train": train_df["dominant_class"].value_counts(normalize=True),
    "val": val_df["dominant_class"].value_counts(normalize=True),
    "test": test_df["dominant_class"].value_counts(normalize=True)
}).fillna(0)

dominant_dist

,train,val,test
dominant_class,,,
Grass,0.470375,0.470,0.470
Tree,0.303750,0.304,0.303
Crop,0.180250,0.181,0.180
Barren,0.019250,0.019,0.020
Water,0.017750,0.018,0.018
Built-up,0.005875,0.006,0.006
Shrub,0.002750,0.002,0.003


Transforms

In [16]:
IMG_SIZE = CONFIG["img_size"]

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

Dataset Classes

In [17]:
class LandCoverDataset(Dataset):
    def __init__(self, df, image_dir, transform=None, tokenizer=None, text_col=None, max_len=128):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.tokenizer = tokenizer
        self.text_col = text_col
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.image_dir, row["filename"])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        target = torch.tensor(
            row[TARGET_COLS].values.astype(np.float32),
            dtype=torch.float32
        )

        item = {
            "image": image,
            "target": target
        }

        if self.tokenizer is not None and self.text_col is not None:
            text = str(row[self.text_col])

            enc = self.tokenizer(
                text,
                padding="max_length",
                truncation=True,
                max_length=self.max_len,
                return_tensors="pt"
            )

            item["input_ids"] = enc["input_ids"].squeeze(0)
            item["attention_mask"] = enc["attention_mask"].squeeze(0)

        return item

In [18]:
class LandCoverRawTextDataset(Dataset):
    def __init__(self, df, image_dir, transform=None, text_col=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.text_col = text_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.image_dir, row["filename"])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        text = str(row[self.text_col])

        target = torch.tensor(
            row[TARGET_COLS].values.astype(np.float32),
            dtype=torch.float32
        )

        return {
            "image": image,
            "text": text,
            "target": target
        }

Swin image-only

In [19]:
class ImageOnlyModel(nn.Module):
    def __init__(self, model_name, num_outputs=7):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0
        )

        image_dim = self.backbone.num_features

        self.regressor = nn.Sequential(
            nn.LayerNorm(image_dim),
            nn.Linear(image_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, num_outputs)
        )

    def forward(self, image):
        image_features = self.backbone(image)
        return self.regressor(image_features)

Swin + MiniLM

In [20]:
class ImageTextGatedFrozenScaledModel(nn.Module):
    def __init__(
        self,
        image_model_name,
        text_model_name,
        num_outputs=7,
        text_scale=0.7
    ):
        super().__init__()

        self.text_scale = text_scale

        self.image_backbone = timm.create_model(
            image_model_name,
            pretrained=True,
            num_classes=0
        )

        image_dim = self.image_backbone.num_features

        self.text_backbone = AutoModel.from_pretrained(text_model_name)

        for p in self.text_backbone.parameters():
            p.requires_grad = False

        text_dim = self.text_backbone.config.hidden_size

        self.text_proj = nn.Linear(text_dim, image_dim)

        self.gate_layer = nn.Sequential(
            nn.Linear(image_dim + image_dim, image_dim),
            nn.Sigmoid()
        )

        self.regressor = nn.Sequential(
            nn.LayerNorm(image_dim),
            nn.Linear(image_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, num_outputs)
        )

    def mean_pooling(self, model_output, attention_mask):
        token_embeddings = model_output.last_hidden_state
        mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

        return torch.sum(token_embeddings * mask, dim=1) / torch.clamp(
            mask.sum(dim=1),
            min=1e-9
        )

    def forward(self, image, input_ids, attention_mask):
        image_features = self.image_backbone(image)

        with torch.no_grad():
            text_output = self.text_backbone(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        text_features = self.mean_pooling(text_output, attention_mask)
        text_features = self.text_proj(text_features)

        gate_input = torch.cat([image_features, text_features], dim=1)
        gate = self.gate_layer(gate_input)

        fused_features = image_features + self.text_scale * gate * text_features

        return self.regressor(fused_features)

Evaluation functions

In [21]:
@torch.no_grad()
def evaluate_image_model(model, loader, config):
    model.eval()

    criterion = nn.L1Loss()
    total_loss = 0
    all_preds = []
    all_targets = []

    for batch in loader:
        images = batch["image"].to(config["device"])
        targets = batch["target"].to(config["device"])

        preds = model(images)
        loss = criterion(preds, targets)

        total_loss += loss.item() * images.size(0)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    class_mae = np.mean(np.abs(y_true - y_pred), axis=0)

    return total_loss / len(loader.dataset), mae, rmse, class_mae

In [22]:
@torch.no_grad()
def evaluate_text_model(model, loader, config):
    model.eval()

    criterion = nn.L1Loss()
    total_loss = 0
    all_preds = []
    all_targets = []

    for batch in loader:
        images = batch["image"].to(config["device"])
        input_ids = batch["input_ids"].to(config["device"])
        attention_mask = batch["attention_mask"].to(config["device"])
        targets = batch["target"].to(config["device"])

        preds = model(images, input_ids, attention_mask)
        loss = criterion(preds, targets)

        total_loss += loss.item() * images.size(0)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    class_mae = np.mean(np.abs(y_true - y_pred), axis=0)

    return total_loss / len(loader.dataset), mae, rmse, class_mae

In [23]:
@torch.no_grad()
def evaluate_remoteclip_model(model, loader, config):
    model.eval()

    criterion = nn.L1Loss()
    total_loss = 0
    all_preds = []
    all_targets = []

    for batch in loader:
        images = batch["image"].to(config["device"])
        texts = batch["text"]
        targets = batch["target"].to(config["device"])

        preds = model(images, texts)
        loss = criterion(preds, targets)

        total_loss += loss.item() * images.size(0)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    class_mae = np.mean(np.abs(y_true - y_pred), axis=0)

    return total_loss / len(loader.dataset), mae, rmse, class_mae

W&B Training Functions

In [24]:
def train_image_model(model, train_loader, val_loader, config):
    criterion = nn.L1Loss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"]
    )

    best_val_mae = float("inf")
    best_state = None
    patience_counter = 0
    history = []

    for epoch in range(config["epochs"]):
        model.train()
        train_loss = 0

        for batch in train_loader:
            optimizer.zero_grad()

            images = batch["image"].to(config["device"])
            targets = batch["target"].to(config["device"])

            preds = model(images)
            loss = criterion(preds, targets)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
            optimizer.step()

            train_loss += loss.item() * images.size(0)

        train_loss /= len(train_loader.dataset)

        val_loss, val_mae, val_rmse, _ = evaluate_image_model(model, val_loader, config)

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse
        })

        print(
            f"Epoch {epoch+1}/{config['epochs']} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val MAE: {val_mae:.4f} | "
            f"Val RMSE: {val_rmse:.4f}"
        )

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse
        })

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= config["early_stopping_patience"]:
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_state)

    return model, pd.DataFrame(history)

In [25]:
def train_text_model(model, train_loader, val_loader, config):
    criterion = nn.L1Loss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"]
    )

    best_val_mae = float("inf")
    best_state = None
    patience_counter = 0
    history = []

    for epoch in range(config["epochs"]):
        model.train()
        train_loss = 0

        for batch in train_loader:
            optimizer.zero_grad()

            images = batch["image"].to(config["device"])
            input_ids = batch["input_ids"].to(config["device"])
            attention_mask = batch["attention_mask"].to(config["device"])
            targets = batch["target"].to(config["device"])

            preds = model(images, input_ids, attention_mask)
            loss = criterion(preds, targets)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
            optimizer.step()

            train_loss += loss.item() * images.size(0)

        train_loss /= len(train_loader.dataset)

        val_loss, val_mae, val_rmse, _ = evaluate_text_model(model, val_loader, config)

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse
        })

        print(
            f"Epoch {epoch+1}/{config['epochs']} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val MAE: {val_mae:.4f} | "
            f"Val RMSE: {val_rmse:.4f}"
        )

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse
        })

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= config["early_stopping_patience"]:
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_state)

    return model, pd.DataFrame(history)

In [26]:
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ayseatci00 (ayseatci00-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

Image Only Model

In [27]:
def run_swin_image_only():
    seed_everything(CONFIG["seed"])

    run_config = CONFIG.copy()
    run_config["experiment"] = "swinv2_image_only"
    run_config["fusion"] = "none"
    run_config["text_column"] = "none"
    run_config["text_model"] = "none"

    wandb.init(
        project="DI725-Project-landcover-regression_phase2_experiments",
        name="swinv2_image_only",
        config=run_config,
        reinit=True
    )

    train_ds = LandCoverDataset(train_df, IMAGE_DIR, transform=train_tfms)
    val_ds = LandCoverDataset(val_df, IMAGE_DIR, transform=eval_tfms)
    test_ds = LandCoverDataset(test_df, IMAGE_DIR, transform=eval_tfms)

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)

    model = ImageOnlyModel(
        model_name=CONFIG["model_name"],
        num_outputs=len(TARGET_COLS)
    ).to(CONFIG["device"])

    model, history = train_image_model(model, train_loader, val_loader, CONFIG)

    test_loss, test_mae, test_rmse, class_mae = evaluate_image_model(model, test_loader, CONFIG)

    wandb.log({
        "test_loss": test_loss,
        "test_mae": test_mae,
        "test_rmse": test_rmse
    })

    wandb.log({
        "class_mae_table": wandb.Table(
            dataframe=pd.DataFrame({
                "class": TARGET_COLS,
                "class_mae": class_mae
            })
        )
    })

    wandb.finish()

    return {
      "experiment": "swinv2_image_only",
      "test_mae": test_mae,
      "test_rmse": test_rmse,
      "text_column": None,
      "text_model": None,
      "text_scale": None,
      "class_mae": class_mae
      }

In [28]:
def run_swin_transformer_text(text_col, text_model, text_scale=0.7):
    seed_everything(CONFIG["seed"])

    run_config = CONFIG.copy()
    run_config["experiment"] = "swinv2_transformer_text"
    run_config["text_column"] = text_col
    run_config["text_model"] = text_model
    run_config["text_scale"] = text_scale
    run_config["fusion"] = "gated_frozen_scaled"

    run_name = f"swinv2_{text_col}_{text_model.split('/')[-1]}_scale_{text_scale}"

    wandb.init(
        project="DI725-Project-landcover-regression_phase2_experiments",
        name=run_name,
        config=run_config,
        reinit=True
    )

    tokenizer = AutoTokenizer.from_pretrained(text_model)

    train_ds = LandCoverDataset(train_df, IMAGE_DIR, transform=train_tfms, tokenizer=tokenizer, text_col=text_col)
    val_ds = LandCoverDataset(val_df, IMAGE_DIR, transform=eval_tfms, tokenizer=tokenizer, text_col=text_col)
    test_ds = LandCoverDataset(test_df, IMAGE_DIR, transform=eval_tfms, tokenizer=tokenizer, text_col=text_col)

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)

    model = ImageTextGatedFrozenScaledModel(
        image_model_name=CONFIG["model_name"],
        text_model_name=text_model,
        num_outputs=len(TARGET_COLS),
        text_scale=text_scale
    ).to(CONFIG["device"])

    model, history = train_text_model(model, train_loader, val_loader, CONFIG)

    test_loss, test_mae, test_rmse, class_mae = evaluate_text_model(model, test_loader, CONFIG)

    wandb.log({
        "test_loss": test_loss,
        "test_mae": test_mae,
        "test_rmse": test_rmse
    })

    wandb.log({
        "class_mae_table": wandb.Table(
            dataframe=pd.DataFrame({
                "class": TARGET_COLS,
                "class_mae": class_mae
            })
        )
    })

    wandb.finish()

    return {
    "experiment": run_name,
    "test_mae": test_mae,
    "test_rmse": test_rmse,
    "text_column": text_col,
    "text_model": text_model,
    "text_scale": text_scale,
    "class_mae": class_mae
      }

Run Experiments

In [29]:
results = []

In [30]:
results.append(run_swin_image_only())
pd.DataFrame(results)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

Epoch 1/15 | Train Loss: 10.9682 | Val MAE: 7.7778 | Val RMSE: 17.7582
Epoch 2/15 | Train Loss: 6.7430 | Val MAE: 4.9107 | Val RMSE: 11.5146
Epoch 3/15 | Train Loss: 5.0898 | Val MAE: 3.5113 | Val RMSE: 8.9017
Epoch 4/15 | Train Loss: 4.3713 | Val MAE: 3.0955 | Val RMSE: 7.8415
Epoch 5/15 | Train Loss: 4.0087 | Val MAE: 2.9009 | Val RMSE: 7.6844
Epoch 6/15 | Train Loss: 3.6747 | Val MAE: 2.7835 | Val RMSE: 7.5142
Epoch 7/15 | Train Loss: 3.4191 | Val MAE: 2.6937 | Val RMSE: 7.2821
Epoch 8/15 | Train Loss: 3.3046 | Val MAE: 2.5172 | Val RMSE: 6.7755
Epoch 9/15 | Train Loss: 3.1178 | Val MAE: 2.5765 | Val RMSE: 6.9001
Epoch 10/15 | Train Loss: 2.9318 | Val MAE: 2.4588 | Val RMSE: 6.2626
Epoch 11/15 | Train Loss: 2.8181 | Val MAE: 2.4280 | Val RMSE: 6.5568
Epoch 12/15 | Train Loss: 2.6824 | Val MAE: 2.3277 | Val RMSE: 6.2206
Epoch 13/15 | Train Loss: 2.5373 | Val MAE: 2.2420 | Val RMSE: 5.9037
Epoch 14/15 | Train Loss: 2.5281 | Val MAE: 2.3798 | Val RMSE: 6.5280
Epoch 15/15 | Train Loss: 

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▅▃▃▂▂▂▂▂▁▁▁▁▁▁
val_loss,█▄▃▂▂▂▂▁▁▁▁▁▁▁▁
val_mae,█▄▃▂▂▂▂▁▁▁▁▁▁▁▁
val_rmse,█▄▃▂▂▂▂▂▂▁▁▁▁▁▁
epoch,15
test_loss,2.4598
test_mae,2.4598


,experiment,test_mae,test_rmse,text_column,text_model,text_scale,class_mae
0,swinv2_image_only,2.459801,6.409931,None,None,None,"[2.8975673, 0.95590156, 6.00242, 4.5041337, 0...."


In [31]:
class_cols = ["Tree", "Shrub", "Grass", "Crop", "Built-up", "Barren", "Water"]

class_results_df = pd.DataFrame([
    {
        "experiment": res["experiment"],
        "text_column": res.get("text_column"),
        "text_model": res.get("text_model"),
        "text_scale": res.get("text_scale"),
        **{f"{c}_mae": v for c, v in zip(class_cols, res["class_mae"])}
    }
    for res in results
])

class_results_df.sort_values("experiment")

,experiment,text_column,text_model,text_scale,Tree_mae,Shrub_mae,Grass_mae,Crop_mae,Built-up_mae,Barren_mae,Water_mae
0,swinv2_image_only,None,None,None,2.897567,0.955902,6.00242,4.504134,0.422388,2.101204,0.334996


MiniLM Experiments

In [32]:
TEXT_COLS_CLEAN = [
    "hybrid_gemma3_4b_clean",
    "text_qwen3_4b_clean",
    "vision_gemma3_4b_clean"
]

In [33]:
for text_col in TEXT_COLS_CLEAN:
    results.append(
        run_swin_transformer_text(
            text_col=text_col,
            text_model="sentence-transformers/all-MiniLM-L6-v2",
            text_scale=0.7
        )
    )

pd.DataFrame(results).sort_values("test_mae")

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/15 | Train Loss: 10.7019 | Val MAE: 7.7088 | Val RMSE: 17.1617
Epoch 2/15 | Train Loss: 6.5130 | Val MAE: 4.5790 | Val RMSE: 10.7647
Epoch 3/15 | Train Loss: 4.8933 | Val MAE: 3.4226 | Val RMSE: 8.6448
Epoch 4/15 | Train Loss: 4.2809 | Val MAE: 3.0283 | Val RMSE: 8.0525
Epoch 5/15 | Train Loss: 3.9847 | Val MAE: 3.3083 | Val RMSE: 8.2537
Epoch 6/15 | Train Loss: 3.6974 | Val MAE: 2.7549 | Val RMSE: 7.4417
Epoch 7/15 | Train Loss: 3.4987 | Val MAE: 2.9726 | Val RMSE: 7.6486
Epoch 8/15 | Train Loss: 3.2887 | Val MAE: 2.6225 | Val RMSE: 6.6212
Epoch 9/15 | Train Loss: 3.1244 | Val MAE: 2.5437 | Val RMSE: 6.6837
Epoch 10/15 | Train Loss: 2.9833 | Val MAE: 2.4571 | Val RMSE: 6.4876
Epoch 11/15 | Train Loss: 2.8412 | Val MAE: 2.4594 | Val RMSE: 6.2350
Epoch 12/15 | Train Loss: 2.6783 | Val MAE: 2.5017 | Val RMSE: 6.6854
Epoch 13/15 | Train Loss: 2.5882 | Val MAE: 2.3927 | Val RMSE: 6.1086
Epoch 14/15 | Train Loss: 2.4815 | Val MAE: 2.1080 | Val RMSE: 5.6478
Epoch 15/15 | Train Loss: 

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▄▃▃▂▂▂▂▂▂▁▁▁▁▁
val_loss,█▄▃▂▃▂▂▂▂▁▁▁▁▁▁
val_mae,█▄▃▂▃▂▂▂▂▁▁▁▁▁▁
val_rmse,█▄▃▂▃▂▂▂▂▂▁▂▁▁▁
epoch,15
test_loss,2.54936
test_mae,2.54936


,experiment,test_mae,test_rmse,text_column,text_model,text_scale,class_mae
2,swinv2_text_qwen3_4b_clean_all-MiniLM-L6-v2_sc...,2.031662,4.670212,text_qwen3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,"[2.5233533, 0.9681544, 5.4953628, 2.6557555, 0..."
1,swinv2_hybrid_gemma3_4b_clean_all-MiniLM-L6-v2...,2.116787,5.104050,hybrid_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,"[2.4822881, 0.9681931, 5.2909665, 3.37878, 0.4..."
0,swinv2_image_only,2.459801,6.409931,None,None,NaN,"[2.8975673, 0.95590156, 6.00242, 4.5041337, 0...."
3,swinv2_vision_gemma3_4b_clean_all-MiniLM-L6-v2...,2.549358,6.699489,vision_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,"[3.2047188, 0.95118237, 6.2111382, 4.6574025, ..."


In [34]:
class_cols = ["Tree", "Shrub", "Grass", "Crop", "Built-up", "Barren", "Water"]

class_results_df = pd.DataFrame([
    {
        "experiment": res["experiment"],
        "text_column": res.get("text_column"),
        "text_model": res.get("text_model"),
        "text_scale": res.get("text_scale"),
        **{f"{c}_mae": v for c, v in zip(class_cols, res["class_mae"])}
    }
    for res in results
])

class_results_df.sort_values("experiment")

,experiment,text_column,text_model,text_scale,Tree_mae,Shrub_mae,Grass_mae,Crop_mae,Built-up_mae,Barren_mae,Water_mae
1,swinv2_hybrid_gemma3_4b_clean_all-MiniLM-L6-v2...,hybrid_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.482288,0.968193,5.290967,3.378780,0.489119,1.866251,0.341909
0,swinv2_image_only,None,None,NaN,2.897567,0.955902,6.002420,4.504134,0.422388,2.101204,0.334996
2,swinv2_text_qwen3_4b_clean_all-MiniLM-L6-v2_sc...,text_qwen3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.523353,0.968154,5.495363,2.655756,0.487932,1.786818,0.304259
3,swinv2_vision_gemma3_4b_clean_all-MiniLM-L6-v2...,vision_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,3.204719,0.951182,6.211138,4.657403,0.448073,2.017154,0.355837


DistilBERT Experiments

In [35]:
CONFIG["text_model"] = "distilbert-base-uncased"

In [36]:
for text_col in TEXT_COLS_CLEAN:
    results.append(
        run_swin_transformer_text(
            text_col=text_col,
            text_model="distilbert-base-uncased",
            text_scale=0.7
        )
    )

pd.DataFrame(results).sort_values("test_mae")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/15 | Train Loss: 10.9806 | Val MAE: 7.8313 | Val RMSE: 17.6638
Epoch 2/15 | Train Loss: 6.4614 | Val MAE: 4.6391 | Val RMSE: 11.3020
Epoch 3/15 | Train Loss: 4.9157 | Val MAE: 3.9962 | Val RMSE: 10.0090
Epoch 4/15 | Train Loss: 4.3291 | Val MAE: 3.1119 | Val RMSE: 8.5687
Epoch 5/15 | Train Loss: 3.9005 | Val MAE: 2.9059 | Val RMSE: 7.9091
Epoch 6/15 | Train Loss: 3.7525 | Val MAE: 2.9111 | Val RMSE: 7.7562
Epoch 7/15 | Train Loss: 3.3952 | Val MAE: 2.7676 | Val RMSE: 7.3242
Epoch 8/15 | Train Loss: 3.2004 | Val MAE: 2.6248 | Val RMSE: 7.0333
Epoch 9/15 | Train Loss: 3.0754 | Val MAE: 2.5148 | Val RMSE: 6.9536
Epoch 10/15 | Train Loss: 2.8661 | Val MAE: 2.5464 | Val RMSE: 6.4636
Epoch 11/15 | Train Loss: 2.7624 | Val MAE: 2.7835 | Val RMSE: 7.5731
Epoch 12/15 | Train Loss: 2.6488 | Val MAE: 2.2662 | Val RMSE: 5.8506
Epoch 13/15 | Train Loss: 2.5173 | Val MAE: 2.3519 | Val RMSE: 6.1739
Epoch 14/15 | Train Loss: 2.3764 | Val MAE: 2.2354 | Val RMSE: 5.8461
Epoch 15/15 | Train Loss:

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▄▃▃▂▂▂▂▂▁▁▁▁▁▁
val_loss,█▄▃▂▂▂▂▁▁▁▂▁▁▁▁
val_mae,█▄▃▂▂▂▂▁▁▁▂▁▁▁▁
val_rmse,█▄▃▃▂▂▂▂▂▁▂▁▁▁▁
epoch,15
test_loss,2.51968
test_mae,2.51968


,experiment,test_mae,test_rmse,text_column,text_model,text_scale,class_mae
5,swinv2_text_qwen3_4b_clean_distilbert-base-unc...,1.787415,4.278047,text_qwen3_4b_clean,distilbert-base-uncased,0.7,"[2.4162302, 0.96036917, 4.10475, 2.573656, 0.4..."
2,swinv2_text_qwen3_4b_clean_all-MiniLM-L6-v2_sc...,2.031662,4.670212,text_qwen3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,"[2.5233533, 0.9681544, 5.4953628, 2.6557555, 0..."
1,swinv2_hybrid_gemma3_4b_clean_all-MiniLM-L6-v2...,2.116787,5.104050,hybrid_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,"[2.4822881, 0.9681931, 5.2909665, 3.37878, 0.4..."
4,swinv2_hybrid_gemma3_4b_clean_distilbert-base-...,2.237545,5.958944,hybrid_gemma3_4b_clean,distilbert-base-uncased,0.7,"[2.6500793, 0.96253157, 5.3718905, 3.8895268, ..."
0,swinv2_image_only,2.459801,6.409931,None,None,NaN,"[2.8975673, 0.95590156, 6.00242, 4.5041337, 0...."
6,swinv2_vision_gemma3_4b_clean_distilbert-base-...,2.519681,6.569022,vision_gemma3_4b_clean,distilbert-base-uncased,0.7,"[3.2409098, 0.9659413, 6.2423916, 4.31593, 0.4..."
3,swinv2_vision_gemma3_4b_clean_all-MiniLM-L6-v2...,2.549358,6.699489,vision_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,"[3.2047188, 0.95118237, 6.2111382, 4.6574025, ..."


In [38]:
class_cols = ["Tree", "Shrub", "Grass", "Crop", "Built-up", "Barren", "Water"]

class_results_df = pd.DataFrame([
    {
        "experiment": res["experiment"],
        "text_column": res.get("text_column"),
        "text_model": res.get("text_model"),
        "text_scale": res.get("text_scale"),
        **{f"{c}_mae": v for c, v in zip(class_cols, res["class_mae"])}
    }
    for res in results
])

class_results_df.sort_values("experiment")

,experiment,text_column,text_model,text_scale,Tree_mae,Shrub_mae,Grass_mae,Crop_mae,Built-up_mae,Barren_mae,Water_mae
1,swinv2_hybrid_gemma3_4b_clean_all-MiniLM-L6-v2...,hybrid_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.482288,0.968193,5.290967,3.378780,0.489119,1.866251,0.341909
4,swinv2_hybrid_gemma3_4b_clean_distilbert-base-...,hybrid_gemma3_4b_clean,distilbert-base-uncased,0.7,2.650079,0.962532,5.371891,3.889527,0.375132,1.984778,0.428875
0,swinv2_image_only,None,None,NaN,2.897567,0.955902,6.002420,4.504134,0.422388,2.101204,0.334996
2,swinv2_text_qwen3_4b_clean_all-MiniLM-L6-v2_sc...,text_qwen3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.523353,0.968154,5.495363,2.655756,0.487932,1.786818,0.304259
5,swinv2_text_qwen3_4b_clean_distilbert-base-unc...,text_qwen3_4b_clean,distilbert-base-uncased,0.7,2.416230,0.960369,4.104750,2.573656,0.410273,1.726926,0.319700
3,swinv2_vision_gemma3_4b_clean_all-MiniLM-L6-v2...,vision_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,3.204719,0.951182,6.211138,4.657403,0.448073,2.017154,0.355837
6,swinv2_vision_gemma3_4b_clean_distilbert-base-...,vision_gemma3_4b_clean,distilbert-base-uncased,0.7,3.240910,0.965941,6.242392,4.315930,0.477669,2.024950,0.369972
